# Chapter 6 (part 1) — K-Nearest Neighbors: Can a Flexible Model Honestly Beat Logistic?

Covers: K-Nearest Neighbors (distance metrics, one-hot encoding, standardization / z-scores, choosing K, KNN as a feature engine), on **churn**.

**Chapter 6 project question:** *can more flexible models honestly beat the Chapter 5 logistic baseline?* The data is the IBM Telco churn table you know: 7,043 customers, 26.5% churned, one fixed stratified split (4,930 train / 2,113 test). The baseline is your notebook-14 logistic model at **test AUC 0.843**. "Honestly" has a precise meaning here: every choice (scaling, K, which score to stack) is made with cross-validation on the **training** set, and the test set is touched once per model.

Setup rebuilds the baseline. KNN's inputs are new, so §1 is a feature step again. The exam goes in a new review notebook, `reviews/R03_knn-and-feature-engines.ipynb` (Q1–Q3; ★Q1 is the one to do).

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import roc_auc_score

pd.set_option("display.max_columns", None)

from mlstats.data import load_churn_split
from mlstats.features import ADDONS, add_churn_features, add_logit_features

train, test = load_churn_split()
train_f = add_logit_features(add_churn_features(train))   # your notebook-13 + notebook-14 features
test_f = add_logit_features(add_churn_features(test))
y_train, y_test = train_f["churned"].to_numpy(), test_f["churned"].to_numpy()

# your notebook-14 logistic model, rebuilt: the baseline every Chapter 6 model has to beat
BASE = 'churned ~ tenure + MonthlyCharges + C(Contract, Treatment("Two year")) + InternetService + PaymentMethod + PaperlessBilling + SeniorCitizen'
FORMULA = BASE + " + " + " + ".join(c + "_yes" for c in ADDONS)
logit_fixed = smf.logit(FORMULA, data=train_f).fit(disp=0)
p_logit = logit_fixed.predict(test_f).to_numpy()
AUC_BASELINE = roc_auc_score(y_test, p_logit)

# one fixed set of CV folds for every cross-validation in this notebook
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

print(f"baseline logistic test AUC {AUC_BASELINE:.4f} | train {len(y_train)}, test {len(y_test)}")

## 1. Build Your Features: What Number Is a Contract?

Covers: *Distance Metrics* and *One-Hot Encoder* (book). KNN calls two customers "neighbors" when the distance between their rows is small, so every column must be a number, and **the number you pick decides how far apart categories sit**. Numeric picture: three customers identical except for contract, A month-to-month, B one-year, C two-year.

| coding | A | B | C | gap A→B | gap B→C |
|---|---|---|---|---|---|
| 0 / 1 / 2 | 0 | 1 | 2 | 1 | 1 |
| months: 1 / 12 / 24 | 1 | 12 | 24 | 11 | 12 |

Both put B halfway between A and C. You can also invent your own monotone coding (a log, say). Write one `add_knn_features(df)` function with two features: **`contract_num`** (your numeric coding of `Contract`) and **`is_fiber`** (1 if `InternetService == "Fiber optic"`, the high-churn group from notebook 14).

The two feature rules still apply. Nothing is computed from the test set, and everything is known at prediction time. Both of these are fixed rules on a customer's own row, so you are safe.

**Predict first:** (a) which churn-rate gap is bigger, month-to-month → one-year or one-year → two-year, and is it about the same, about 2×, or about 4× as big? (b) Once columns are z-scored (§2), will swapping your coding for another reasonable one move test AUC by under 0.005, 0.005–0.02, or over 0.02?

*Tools for this question:*
- `df["Contract"].map({...})` takes a dict. The three levels are exactly `"Month-to-month"`, `"One year"` and `"Two year"`. Gotcha: a level missing from the dict becomes `NaN`, and KNN cannot take `NaN`.
- `np.log(...)` for a log coding; `(cond).astype(int)` for a 0/1 flag.

_Prediction (one line, before running anything):_
* 

In [ ]:
def add_knn_features(df):
    """KNN features: contract_num and is_fiber. Returns a copy."""
    out = df.copy()
    out["contract_num"] = ...   # your numeric coding of Contract
    out["is_fiber"] = ...       # 1 if InternetService == "Fiber optic", else 0
    return out

In [ ]:
# pre-filled: consumes your add_knn_features
train_k, test_k = add_knn_features(train_f), add_knn_features(test_f)

print(train_k.groupby("Contract").agg(churn_rate=("churned", "mean"), contract_num=("contract_num", "first")).round(3), "\n")

def z_knn_auc(Xtr, Xte, k=25):
    """Test AUC of a z-scored KNN (scaler fit on train inside the pipeline)."""
    m = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k)).fit(Xtr, y_train)
    return roc_auc_score(y_test, m.predict_proba(Xte)[:, 1])

codings = {"yours": None,
           "0/1/2": {"Month-to-month": 0, "One year": 1, "Two year": 2},
           "months 1/12/24": {"Month-to-month": 1, "One year": 12, "Two year": 24},
           "log months": {"Month-to-month": 0.0, "One year": np.log(12), "Two year": np.log(24)}}
aucs = {}
for name, mapping in codings.items():
    parts = []
    for d, df_f in ((train_k, train_f), (test_k, test_f)):
        cn = d["contract_num"] if mapping is None else df_f["Contract"].map(mapping)
        parts.append(d[["tenure", "MonthlyCharges", "is_fiber"]].assign(contract_num=cn))
    aucs[name] = z_knn_auc(*parts)
print("z-scored KNN (K=25) test AUC by contract coding:", {k: round(v, 4) for k, v in aucs.items()})

_What matched, what surprised you, and why (2-3 bullets):_
* 

## 2. Neighbors by Hand, and Why Scale Matters

Covers: *K-Nearest Neighbors*, *Distance Metrics*, *Standardization (Normalization, Z-Scores)* (book). KNN predicts a customer's churn as the average outcome of the K customers **closest** to them, where closest = smallest Euclidean distance (square root of the sum of squared per-column gaps). Numeric picture, not churn: person A is 170 cm tall and earns \$50,000, B is 190 cm and earns \$50,100, C is 171 cm and earns \$60,000. A–B squared distance = 20² + 100² = 10,400 (distance 102); A–C = 1² + 10,000² (distance ≈ 10,000). A's "nearest" person is B, 20 cm taller, because dollars are bigger numbers than centimeters. Height never got a vote. **Z-scores** fix the units: subtract the training mean and divide by the training standard deviation, so "1 unit" means "1 standard deviation" in every column.

(a) **By hand, once:** z-score the five KNN columns using train statistics only.
(b) **By hand, once:** write `knn_prob` (the mean outcome of the k nearest rows) and run it for test customer 0, in raw space and in z space, K = 25.
(c) **Library:** fit `KNeighborsClassifier` on raw and on z-scored data, and compare test AUC to the baseline.

(A column's share of the total squared distance equals its variance divided by the sum of all the columns' variances; the pre-filled cell uses this.)

**Predict first:** (a) in raw units, which of the five columns own **over 90%** of the squared distance, and which own **under 1%**? (b) Will z-scoring raise test AUC by under 0.01, 0.01–0.03, or over 0.03?

*Tools for this question:*
- `X.mean(axis=0)` and `X.std(axis=0)` give per-column statistics of a NumPy matrix; NumPy broadcasting lets `(X - mu) / sd` work with no loop. Test rows use the **train** `mu` and `sd` (gotcha: never recompute them on test).
- Distances from one row `x` to every row of `X`: `np.sqrt(((X - x) ** 2).sum(axis=1))`. `np.argsort(d)[:k]` gives the positions of the k smallest. `y[idx].mean()` is the prediction.
- `KNeighborsClassifier(n_neighbors=25).fit(X, y)`, then `.predict_proba(X_new)[:, 1]`.
- Ties: duplicate customers sit at exactly the same distance. If your answer differs from the library's by one neighbor, that is tie-breaking, not a bug (customer 0 has no such tie).

_Prediction (one line, before running anything):_
* 

In [ ]:
# pre-filled: the five KNN columns as NumPy matrices (uses your add_knn_features)
KNN_COLS = ["tenure", "MonthlyCharges", "TotalCharges", "contract_num", "is_fiber"]
X_train_raw = train_k[KNN_COLS].to_numpy(float)
X_test_raw = test_k[KNN_COLS].to_numpy(float)

In [ ]:
mu = ...        # train mean of each column
sd = ...        # train standard deviation of each column
Z_train = ...   # z-scores of the training rows
Z_test = ...    # z-scores of the test rows, using the TRAIN mu and sd

In [ ]:
# pre-filled: consumes your Z_train
raw_share = X_train_raw.var(axis=0) / X_train_raw.var(axis=0).sum()
z_share = Z_train.var(axis=0) / Z_train.var(axis=0).sum()
print("share of squared distance:")
print(pd.DataFrame({"raw units": raw_share, "z-scored": z_share}, index=KNN_COLS).round(6))
print("check on Z_train: means", Z_train.mean(axis=0).round(3), "| sds", Z_train.std(axis=0).round(3))

In [ ]:
def knn_prob(X, y, x, k=25):
    """P(churn) for one query row x: the mean of y over the k rows of X closest to x (Euclidean)."""
    ...

p_hand_raw = knn_prob(X_train_raw, y_train, X_test_raw[0])
p_hand_z = knn_prob(Z_train, y_train, Z_test[0])

In [ ]:
# pre-filled: who are test customer 0's neighbors in each space?
q = test_k.iloc[0]
print(f"customer 0: tenure {q['tenure']}, contract_num {q['contract_num']}, fiber {q['is_fiber']}, actually churned: {y_test[0]}\n")
rows = []
for name, X, x, p in (("raw", X_train_raw, X_test_raw[0], p_hand_raw), ("z", Z_train, Z_test[0], p_hand_z)):
    idx = np.argsort(np.sqrt(((X - x) ** 2).sum(axis=1)))[:25]
    nb = train_k.iloc[idx]
    rows.append({"space": name, "P(churn) by hand": p, "neighbors' mean tenure": nb["tenure"].mean(),
                 "neighbors' mean contract_num": nb["contract_num"].mean(), "neighbors' share fiber": nb["is_fiber"].mean()})
print(pd.DataFrame(rows).round(3).to_string(index=False))

In [ ]:
knn_raw = ...   # KNN, k=25, fit on the raw training matrix
knn_z = ...     # KNN, k=25, fit on the z-scored training matrix

In [ ]:
# pre-filled: consumes knn_raw, knn_z
auc_raw = roc_auc_score(y_test, knn_raw.predict_proba(X_test_raw)[:, 1])
auc_z = roc_auc_score(y_test, knn_z.predict_proba(Z_test)[:, 1])
print(f"customer 0, hand vs library (z): {p_hand_z:.3f} vs {knn_z.predict_proba(Z_test[[0]])[0, 1]:.3f}")
print(f"test AUC: raw {auc_raw:.3f} | z-scored {auc_z:.3f} | logistic baseline {AUC_BASELINE:.3f}")

_What matched, what surprised you, and why (2-3 bullets):_
* 

## 3. Choosing K: Memorizing vs. Averaging

Covers: *Choosing K* (book). Numeric picture: a customer's 5 nearest training neighbors churned 1, 0, 0, 1, 0, so K = 5 predicts 2/5 = 0.4. With K = 1 the prediction is the single nearest neighbor's label, so 1.0 here, and one different neighbor flips it to 0.0. That is variance: small K chases noise. Large K averages over customers who are not that similar to you, which is bias: at K = n, everyone gets the base rate.

(a) **The one heavy step:** loop over `K_GRID` and compute the mean 5-fold CV AUC on the **training set only**, with the scaler inside the pipeline.
(b) Pick `best_k` from your numbers. A pre-filled cell then scores the test set **once**.

**Predict first:** (a) at K = 1, will the in-sample (train-on-train) AUC be above 0.95, 0.85–0.95, or below 0.85, and will the 5-fold CV AUC be below 0.70, 0.70–0.80, or above 0.80? (b) Does the CV curve peak at K below 10, between 10 and 30, or above 30?

*Tools for this question:*
- `cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k)), X_train_raw, y_train, scoring="roc_auc", cv=skf)` returns an **array of 5 scores**; take `.mean()`.
- Pass the **raw** `X_train_raw`, not `Z_train`: the pipeline then re-fits the scaler on each fold's training part. Z-scoring everything first would let each validation fold leak into the mean and sd.

_Prediction (one line, before running anything):_
* 

In [ ]:
K_GRID = [1, 3, 5, 10, 25, 50, 75, 100, 150, 200, 300]   # pre-filled grid

In [ ]:
cv_auc = ...   # dict {k: mean 5-fold CV ROC AUC on the TRAINING set} for every k in K_GRID

In [ ]:
# pre-filled: in-sample AUC next to your CV AUC
train_auc = {k: roc_auc_score(y_train, make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
                              .fit(X_train_raw, y_train).predict_proba(X_train_raw)[:, 1]) for k in K_GRID}
print(pd.DataFrame({"in-sample AUC": train_auc, "5-fold CV AUC": cv_auc}).round(3).T.to_string())

fig, ax = plt.subplots(figsize=(6, 3.8))
ax.plot(K_GRID, [train_auc[k] for k in K_GRID], "o-", label="in-sample (train on train)")
ax.plot(K_GRID, [cv_auc[k] for k in K_GRID], "o-", label="5-fold CV (honest)")
ax.axhline(AUC_BASELINE, ls="--", color="gray", label="logistic baseline (test)")
ax.set_xscale("log"); ax.set_xlabel("K (log scale)"); ax.set_ylabel("AUC"); ax.legend(); ax.set_title("Choosing K", loc="left")
plt.show()

In [ ]:
best_k = ...   # your pick from cv_auc

In [ ]:
# pre-filled: the one and only test-set look at KNN, at your best_k
final = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=best_k)).fit(X_train_raw, y_train)
p_knn = final.predict_proba(X_test_raw)[:, 1]
knn_test_auc = roc_auc_score(y_test, p_knn)
print(f"best_k {best_k}: CV AUC {cv_auc[best_k]:.4f} | test AUC {knn_test_auc:.4f} | logistic baseline {AUC_BASELINE:.4f}")

_Also explain: why is the in-sample AUC at K = 1 so high, and why is it still a bit below 1.0?_

_What matched, what surprised you, and why (2-3 bullets):_
* 

## 4. KNN as a Feature Engine: Don't Let a Row Vote for Itself

Covers: *KNN as a Feature Engine* (book). A KNN score ("what share of this customer's neighbors churned?") can be fed as a new column into another model. Here, into your notebook-14 logistic formula. The trap: in training, a customer's own label is among their neighbors (distance 0). With K = 5 and label 1, one of the five votes is the answer itself, so the score looks better on train than it can ever look on a new customer. The fix is **out-of-fold**: each training row is scored by a KNN fit on the *other four folds only*, so no row ever votes for itself. The test score still comes from a KNN fit on the whole training set.

(a) Build `knn_oof` (out-of-fold train scores at `best_k`) and `knn_test`.
(b) A pre-filled cell stacks the leaky and the out-of-fold versions into the logistic formula and compares them. You interpret.

**Predict first:** (a) will adding the KNN score move the logistic model's test AUC by under 0.01, 0.01–0.03, or over 0.03 (from 0.843)? (b) Which version gets the larger `knn_score` coefficient, leaky or out-of-fold?

*Tools for this question:*
- `cross_val_predict(make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=best_k)), X_train_raw, y_train, cv=skf, method="predict_proba")` returns an `(n, 2)` array; take `[:, 1]`.
- `knn_test` is the fitted pipeline `final` from §3 applied to `X_test_raw`; `.predict_proba(...)[:, 1]` again.

_Prediction (one line, before running anything):_
* 

In [ ]:
knn_oof = ...    # out-of-fold P(churn) for every training row, at best_k
knn_test = ...   # P(churn) for every test row, from a KNN fit on the whole training set

In [ ]:
# pre-filled: stack both versions into the notebook-14 logistic formula
knn_leaky = final.predict_proba(X_train_raw)[:, 1]   # the leaky version: train rows scored by a KNN that saw them

rows = []
for name, s_train in (("leaky (in-sample)", knn_leaky), ("out-of-fold", knn_oof)):
    m = smf.logit(FORMULA + " + knn_score", data=train_f.assign(knn_score=s_train)).fit(disp=0)
    auc = roc_auc_score(y_test, m.predict(test_f.assign(knn_score=knn_test)))
    rows.append({"version": name, "knn_score coef": m.params["knn_score"], "z": m.tvalues["knn_score"],
                 "score AUC on train": roc_auc_score(y_train, s_train), "stacked test AUC": auc})
res = pd.DataFrame(rows).set_index("version")
print(res.round(4).to_string(), "\n")

print("Chapter 6 ledger (test AUC):")
print(pd.Series({"logistic baseline (notebook 14)": AUC_BASELINE,
                 f"KNN alone (k={best_k})": knn_test_auc,
                 "logistic + out-of-fold KNN score": res.loc["out-of-fold", "stacked test AUC"]}).round(4).to_string())

_What matched, what surprised you, and why (2-3 bullets):_
* 